In [3]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="llama3.2:latest")

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [5]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")


Agent with PII middleware created successfully!


In [6]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)
result

=== Agent Response ===
I cannot provide a response that includes your email address or credit card number. Is there anything else I can help you with?


{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='fa8c1d16-3b2a-4bae-8bd5-d6ad42be10a3'),
  AIMessage(content='I cannot provide a response that includes your email address or credit card number. Is there anything else I can help you with?', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-10-30T10:03:39.433528Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16400527700, 'load_duration': 6873065600, 'prompt_eval_count': 175, 'prompt_eval_duration': 5276741000, 'eval_count': 26, 'eval_duration': 4223447000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--01a18c43-9a14-73ff-ae11-ad4df6bcb28b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 175, 'output_tokens': 26, 'total_tokens': 201})]}

In [7]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content
